In [11]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

In [12]:
df = pd.read_excel("Appened Years Dataset.xlsx")

In [13]:
for i, val in enumerate(df["Year"].iloc):
    if df["Year"].iloc[i] != df["Year"].iloc[i-1]+1:
        df.drop(df["Year"].iloc[i])

In [14]:
df = df[df["Year"] == df["Previous Year"] + 1]
df.head()

,Unnamed: 0,ein,Year,Accounts,Contributions,Grants,Assets,Type,subtype,ein_2,Previous Year,Accounts_2,Contributions_2,Grants_2,Assets_2
1,1,10391479,2008,276,8389212.0,7419626,68897739.0,Community Foundation,Community,10391479.0,2007.0,270.0,13347384.0,10628512.0,95282896.0
2,2,10391479,2009,278,10016375.0,8857052,83057576.0,Community Foundation,Community,10391479.0,2008.0,276.0,8389212.0,7419626.0,68897739.0
3,3,10391479,2010,284,14088617.0,7569699,97807493.0,Community Foundation,Community,10391479.0,2009.0,278.0,10016375.0,8857052.0,83057576.0
4,4,10391479,2011,296,19783555.0,9802671,106056922.0,Community Foundation,Community,10391479.0,2010.0,284.0,14088617.0,7569699.0,97807493.0
5,5,10391479,2012,307,11504387.0,9544622,119272982.0,Community Foundation,Community,10391479.0,2011.0,296.0,19783555.0,9802671.0,106056922.0


In [15]:
df = df.drop(["Unnamed: 0"], axis=1)
df = pd.get_dummies(df, columns=["Type"], dtype=int)
df = pd.get_dummies(df, columns=["subtype"],dtype=int)

df = df[df.iloc[:,0] == df.iloc[:,6]].reset_index(drop=True)

In [16]:
mkt = [23.31,24.23,-19.44,26.89,16.26,28.88,-6.24,19.42,9.54,-0.73, 11.39, 29.6, 13.41, 0, 12.78, 23.45, -38.49, 3.53]
mkt = mkt[::-1]
lst = []

# for addition to the spreadsheet
for i in df['Year']:
  i -= 2007
  lst.append(mkt[i])


df["stocks"] = lst

In [17]:
df.columns

Index(['ein', 'Year', 'Accounts', 'Contributions', 'Grants', 'Assets', 'ein_2',
       'Previous Year', 'Accounts_2', 'Contributions_2', 'Grants_2',
       'Assets_2', 'Type_Community Foundation', 'Type_National',
       'Type_Single Issue', 'Type_Single Issue Charity', 'subtype_Community',
       'subtype_Donation Processor', 'subtype_Other',
       'subtype_Religiously-Affiliated', 'subtype_Standard',
       'subtype_Universities and Healthcare', 'stocks'],
      dtype='object')

In [18]:
lower_limit = df['Contributions'].quantile(0.075)
upper_limit = df['Contributions'].quantile(0.925)

df_filtered = df[(df['Contributions'] >= lower_limit) & (df['Contributions'] <= upper_limit)]

# x = df_filtered[['Accounts_2', 'Contributions_2']]
# y = df_filtered["Contributions"]

x = df[['Accounts_2', 'Contributions_2']]
y = df["Contributions"]

scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.25, random_state = 40)

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.fit_transform(x_test)

In [19]:
model = LinearRegression()
# degree = 8
# model = make_pipeline(PolynomialFeatures(degree), LinearRegression())

model.fit(x_train, y_train)

predictions = model.predict(x_test)

def relative_error(actual_col, predicted_col):
    
    predicted_col = pd.Series(predicted_col, index=actual_col.index)

    valid_rows = (~predicted_col.isna())&(~actual_col.isna())&(actual_col != 0)
    total_actual = actual_col[valid_rows]
    total_predicted = predicted_col[valid_rows]
    RE = (abs((total_actual - total_predicted)) / len(total_actual)).median()
    ME =(abs((total_actual - total_predicted)/total_actual)).median()
    return ME

print(relative_error(y_test, predictions))

1.4575142208188854


In [20]:
print(y_test - predictions)

1343   -6.098311e+05
342    -2.168137e+08
5066   -6.441394e+05
3324   -6.210445e+05
5412   -5.998747e+05
            ...     
1932   -8.849816e+05
1999   -5.450768e+05
538    -2.723325e+05
2941   -9.081834e+05
3372   -7.959001e+05
Name: Contributions, Length: 2017, dtype: float64
